# Dataset B: llimllib/nba_data, Playoff Player Stats (2015-16 to 2025-26)

This notebook builds the **playoff** player features from `ListOfPotentialFeatures.md`:

| Feature | Column(s) | Level |
|---|---|---|
| Playoff PPG, RPG, APG, BPG, SPG | `po_ppg`, `po_rpg`, `po_apg`, `po_bpg`, `po_spg` | player-season |
| Playoff true shooting % | `po_ts_pct` | player-season |
| Playoff minutes per game | `po_mpg` | player-season |
| (extra) playoff games played, on-court plus/minus per game | `po_gp`, `po_plus_minus` | player-season |

**Playoff Box +/-** comes from Basketball-Reference in notebook `03`. Notebook `999` combines this table with the regular-season stats (`01`) and Box +/- (`03`).

*Run order: `01` → `02` → `03` → `999`. This notebook reads the raw files downloaded by notebook 01.*

- **Dataset name:** llimllib/nba_data, `player_season_stats` with `season_type = 'playoffs'` (the same table as notebook 01)
- **Link:** https://github.com/llimllib/nba_data
- **Reads (downloaded by notebook 01):** `data-files/00-raw/llimllib_player_season_stats_raw.csv`, `llimllib_players_raw.csv`, `llimllib_team_seasons_raw.csv`

The variables are the same as in notebook 01: season **totals** (`gp`, `min`, `pts`, `reb`, `ast`, `stl`, `blk`, `fga`, `fta`, `plus_minus`), plus `ts_pct`, `age`, and the player's last team, for one player in one postseason.

## Concerns going in

1. **Playoff samples are small.** A team plays 4 to 28 playoff games, and a bench player may appear in only one or two. As in notebook 01, we **keep every row**; `po_gp` and `po_mpg` show how much to trust the per-game numbers.
2. **Are play-in games included?** Play-in games are not playoff games. A player whose team lost in the play-in should have no playoff row. We check this in section 3D.
3. **Only 16 of 30 teams make the playoffs,** so most player-seasons have no playoff row. Their playoff columns will be blank in the combined table (notebook 999). That isn't missing data, since there were no playoff games to record.
4. **Some players may appear only in the playoffs** (signed late, after the regular season ended). Notebook 999 needs their age and experience too, so this table carries `age` and `years_in_nba`, built the same way as in notebook 01.

## Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path('..') / 'salary-data'))
from nba_cleaning_utils import RAW, PROCESSED, FIRST_SEASON, LAST_SEASON, normalize_name

## 1. Load the data (downloaded by notebook 01)

In [2]:
ids = {'player_id': str, 'player_last_team_id': str, 'team_id': str}
stats_raw = pd.read_csv(RAW / 'llimllib_player_season_stats_raw.csv', dtype=ids)
players = pd.read_csv(RAW / 'llimllib_players_raw.csv', dtype=ids)
team_seasons = pd.read_csv(RAW / 'llimllib_team_seasons_raw.csv', dtype=ids)

po_raw = stats_raw[stats_raw['season_type'] == 'playoffs'].copy()
rs_raw = stats_raw[stats_raw['season_type'] == 'regular_season']
po_raw.head()

,season,season_type,player_id,player_last_team_id,team_count,age,gp,min,pts,reb,ast,stl,blk,fga,fta,plus_minus,ts_pct,draft_year
0,1997,playoffs,100,NaN,1,30.0,3,18.481667,1,1,2,0,0,2,2,9,0.174,NaN
1,1997,playoffs,1000,NaN,1,23.0,18,294.965000,83,48,13,11,1,66,28,-30,0.530,1996.0
2,1997,playoffs,1006,NaN,1,39.0,3,22.016667,4,1,0,0,0,5,0,-11,0.400,1981.0
3,1997,playoffs,1037,NaN,1,33.0,1,5.800000,0,1,0,0,0,2,0,-6,0.000,NaN
4,1997,playoffs,1043,NaN,1,23.0,2,6.800000,5,3,1,0,0,3,3,9,0.579,1996.0


## 2A. Is the data tidy?

In [3]:
print('duplicate (season, player_id) playoff rows:', po_raw.duplicated(['season', 'player_id']).sum())
print('playoff rows by number of teams played for:', po_raw['team_count'].value_counts().to_dict())

duplicate (season, player_id) playoff rows: 0
playoff rows by number of teams played for: {1: 6099}


**What we found:** The playoff table is tidy, with one row per player per postseason and no duplicate keys. Every row has `team_count` = 1, as it should, because players can't be traded during the playoffs.

## 2B. Size of the data

In [4]:
po = po_raw.query('@FIRST_SEASON <= season <= @LAST_SEASON').copy()
print('playoff rows, 2015-16 to 2025-26:', len(po), '| different players:', po['player_id'].nunique())
po.groupby('season').size().rename('players').to_frame().T

playoff rows, 2015-16 to 2025-26: 2405 | different players: 864


season,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
players,215,215,210,212,217,239,217,217,214,219,230


In [5]:
in_rs = po.merge(rs_raw[['season', 'player_id']], on=['season', 'player_id'], how='left', indicator=True)
print('playoff rows with a regular-season row the same season:', (in_rs['_merge'] == 'both').sum())
playoff_only = in_rs[in_rs['_merge'] == 'left_only'].merge(players[['player_id', 'name']], on='player_id')
playoff_only[['season', 'name', 'gp', 'min']]

playoff rows with a regular-season row the same season: 2398


,season,name,gp,min
0,2016,John Holland,1,1.316667
1,2016,Dorell Wright,5,18.910000
2,2018,Ty Lawson,5,96.300000
3,2020,Jaylen Adams,3,20.733333
4,2022,Luca Vildoza,7,16.936667
5,2023,DaQuan Jeffries,2,4.588333
6,2023,Tristan Thompson,6,32.341667


**What we found:** 2,405 player-postseasons (864 different players) over 11 seasons, 210-239 a year. That fits 16 playoff teams with 13-15 players each. 2,398 playoff rows have a regular-season row for the same player and season. The other 7 are players who joined a playoff team after the regular season ended (Ty Lawson with Washington in 2018, Tristan Thompson with the Lakers in 2023, ...), most with only a few minutes. This table carries their age and experience so they aren't lost in notebook 999.

## 2C. Missing data

In [6]:
po.isna().sum()

season                   0
season_type              0
player_id                0
player_last_team_id     75
team_count               0
age                      0
gp                       0
min                      0
pts                      0
reb                      0
ast                      0
stl                      0
blk                      0
fga                      0
fta                      0
plus_minus               0
ts_pct                   0
draft_year             376
dtype: int64

In [7]:
no_attempts = po[(po['fga'] + po['fta']) == 0]
print('playoff rows with no shot attempts:', len(no_attempts), '| their ts_pct values:', no_attempts['ts_pct'].value_counts(dropna=False).to_dict())
print('their games played:', sorted(no_attempts['gp'].unique().tolist()))

playoff rows with no shot attempts: 66 | their ts_pct values: {0.0: 66}
their games played: [1, 2, 3, 4, 6]


**What we found:** `draft_year` is blank for undrafted players, and `player_last_team_id` for 75 rows (the source didn't record the team), as in notebook 01. Neither is missing data we need to fill: `last_team` is reference information only. And again, **`ts_pct` is 0 for players with no shot attempts**, where it should be undefined. There are 66 of these, each with 1-6 games, which is more than in the regular season (35), because many end-of-bench players only get garbage-time minutes in the playoffs. We set them to blank in section 3.

## 2D. Outliers and suspicious entries

The checks are the same as notebook 01, with playoff limits: at most **28 games** (4 rounds × 7). We also check that play-in games are excluded, using two stars whose teams lost in the 2023-24 play-in: Stephen Curry (GSW) and DeMar DeRozan (CHI).

In [8]:
po['mpg'] = po['min'] / po['gp']
po['ppg'] = po['pts'] / po['gp']
ts_calc = po['pts'] / (2 * (po['fga'] + 0.44 * po['fta']))

print('rows with a negative stat:', (po[['gp', 'min', 'pts', 'reb', 'ast', 'stl', 'blk', 'fga', 'fta']] < 0).any(axis=1).sum())
print('rows with gp = 0:', (po['gp'] == 0).sum(), '| more than 28 games:', (po['gp'] > 28).sum(), '| max games:', po['gp'].max())
print('rows with more than 48 minutes per game:', (po['mpg'] > 48).sum())
print('rows where ts_pct differs from its formula by more than 0.002:', ((ts_calc - po['ts_pct']).abs() > 0.002).sum())
print('rows with ts_pct above 1:', (po['ts_pct'] > 1).sum(), '| their games played:', sorted(po.loc[po['ts_pct'] > 1, 'gp'].unique().tolist()))

named = po.merge(players[['player_id', 'name']], on='player_id')
for name in ['Stephen Curry', 'DeMar DeRozan']:
    print(f'{name} has a 2023-24 playoff row:', ((named['name'] == name) & (named['season'] == 2024)).any())

rows with a negative stat: 0
rows with gp = 0: 0 | more than 28 games: 0 | max games: 24
rows with more than 48 minutes per game: 0
rows where ts_pct differs from its formula by more than 0.002: 0
rows with ts_pct above 1: 23 | their games played: [1, 2, 3, 5, 6]
Stephen Curry has a 2023-24 playoff row: False
DeMar DeRozan has a 2023-24 playoff row: False


In [9]:
(named.sort_values('ppg', ascending=False).groupby('season').head(1)
      .sort_values('season')[['season', 'name', 'gp', 'ppg', 'mpg']].round(1))

,season,name,gp,ppg,mpg
50,2016,Kevin Durant,18,28.4,40.3
295,2017,Russell Westbrook,5,37.4,38.9
629,2018,LeBron James,22,34.0,41.9
737,2019,Kevin Durant,12,32.2,36.8
889,2020,Donovan Mitchell,7,36.3,37.7
1139,2021,Luka Dončić,7,35.7,40.1
1378,2022,Luka Dončić,15,31.7,36.8
1700,2023,Kawhi Leonard,2,34.5,40.2
1948,2024,Joel Embiid,6,33.0,41.4
2164,2025,Giannis Antetokounmpo,5,33.0,37.6


In [10]:
known_lines = [            # (player, season, stat, published value)
    ('Kawhi Leonard', 2019, 'ppg', 30.5),
    ('Kawhi Leonard', 2019, 'gp', 24),
    ('LeBron James', 2016, 'ppg', 26.3),
    ('LeBron James', 2016, 'gp', 21),
]
for name, season, stat, expected in known_lines:
    got = named.loc[(named['name'] == name) & (named['season'] == season), stat].iloc[0]
    print(f"{'OK   ' if abs(got - expected) <= 0.15 else 'CHECK'} {name} {season} {stat}: got {got:.2f}, published {expected}")

OK    Kawhi Leonard 2019 ppg: got 30.50, published 30.5
OK    Kawhi Leonard 2019 gp: got 24.00, published 24
OK    LeBron James 2016 ppg: got 26.29, published 26.3
OK    LeBron James 2016 gp: got 21.00, published 21


**What we found:**

- **No impossible values.** No stat is negative, nobody has 0 games or more than 48 minutes per game, and nobody has more than 28 games (the most is 24, a team that played four full rounds). `ts_pct` matches its formula on every row.
- **`ts_pct` above 1 appears only for players with a few games** (23 rows, 1-6 games), as in the regular season.
- **Play-in games are not included.** Curry and DeRozan have no 2023-24 playoff row, because their teams lost in the play-in.
- **The top scorer in each postseason is a real star,** and the published stat lines for Kawhi Leonard (2019) and LeBron James (2016) match. Several top lines come from short runs: Russell Westbrook's 37.4 in 2017 was over 5 games, and Kawhi Leonard's 34.5 in 2023 over only 2 (he was injured). So playoff scoring extremes are partly a small-sample effect. They are real performances, so no rows are removed.

## 2E. Regular season vs playoffs: same players

Comparing the two whole tables would be misleading, because the playoff table only has players on good teams. Instead, we pair each player's playoff row with **his own** regular-season row from the same season. For this comparison only, we require at least 10 regular-season games and 4 playoff games, so that tiny samples don't dominate the averages. The saved table keeps every row. **Points per 36 minutes** separates "scores more because he plays more" from "scores more per minute on the floor."

In [11]:
pair_cols = ['gp', 'min', 'pts', 'reb', 'ast', 'ts_pct', 'plus_minus']
paired = (rs_raw.query('gp >= 10')[['season', 'player_id'] + pair_cols]
          .merge(po.query('gp >= 4')[['season', 'player_id'] + pair_cols], on=['season', 'player_id'], suffixes=('_rs', '_po')))
rows = {}
for side in ['rs', 'po']:
    gp = paired[f'gp_{side}']
    rows[side] = {'mpg': paired[f'min_{side}'] / gp, 'ppg': paired[f'pts_{side}'] / gp, 'rpg': paired[f'reb_{side}'] / gp,
                  'apg': paired[f'ast_{side}'] / gp, 'ts_pct': paired[f'ts_pct_{side}'], 'plus_minus': paired[f'plus_minus_{side}'] / gp,
                  'pts_per_36': paired[f'pts_{side}'] / paired[f'min_{side}'] * 36}
comparison = pd.DataFrame({
    'regular season': {k: v.mean() for k, v in rows['rs'].items()},
    'playoffs': {k: v.mean() for k, v in rows['po'].items()},
    '% of players higher in playoffs': {k: (rows['po'][k] > rows['rs'][k]).mean() * 100 for k in rows['rs']},
}).round(3)
print('player-seasons in both:', len(paired))
comparison

player-seasons in both: 1945


,regular season,playoffs,% of players higher in playoffs
mpg,23.557,22.363,46.992
ppg,11.110,9.770,31.362
rpg,4.324,3.940,38.406
apg,2.479,2.042,25.656
ts_pct,0.572,0.541,35.784
plus_minus,1.783,-1.285,28.586
pts_per_36,15.960,14.304,30.180


**What we found:** For 1,945 player-seasons with both, **every stat is lower in the playoffs on average**. Points per game fall from 11.1 to 9.8, and only 31% of players score more. True shooting falls from 57.2% to 54.1%. The drop comes more from **scoring less per minute** (points per 36 minutes fall about 10%, from 16.0 to 14.3) than from playing less (minutes fall about 5%, from 23.6 to 22.4, and 47% of players play more), so rotations shift rather than shrink across the board. On-court plus/minus falls from +1.8 to −1.3, because every playoff opponent is a good team. So the playoff columns describe something different from the regular-season ones, which is a reason to keep both.

*Caveat:* this only covers players whose teams made the playoffs.

## 3. Clean the data

Per-game stats from the totals. `ts_pct` blank for players with no attempts. `years_in_nba` counted from each player's first season in the stats table, with the same edge-case rule as notebook 01 (players who debuted before the source's first season use `draft_year + 1`). Playoff stats get the `po_` prefix.

In [12]:
for stat, total in [('ppg', 'pts'), ('rpg', 'reb'), ('apg', 'ast'), ('bpg', 'blk'), ('spg', 'stl'),
                    ('mpg', 'min'), ('plus_minus', 'plus_minus')]:
    po[stat] = po[total] / po['gp']
po['ts_pct'] = po['ts_pct'].mask((po['fga'] + po['fta']) == 0)

first_season = stats_raw.groupby('player_id')['season'].min()
po['first_nba_season'] = po['player_id'].map(first_season)
truncated = (po['first_nba_season'] == stats_raw['season'].min()) & (po['draft_year'] + 1 < po['first_nba_season'])
po.loc[truncated, 'first_nba_season'] = po.loc[truncated, 'draft_year'] + 1
po['years_in_nba'] = (po['season'] - po['first_nba_season'] + 1).astype(int)

abbrev = team_seasons[['season', 'team_id', 'nba_abbrev']].rename(columns={'team_id': 'player_last_team_id', 'nba_abbrev': 'last_team'})
out = (po.merge(players[['player_id', 'name']], on='player_id', how='left', validate='many_to_one')
         .merge(abbrev, on=['season', 'player_last_team_id'], how='left', validate='many_to_one'))
out['name_key'] = out['name'].map(normalize_name)

po_stats = ['gp', 'mpg', 'ppg', 'rpg', 'apg', 'bpg', 'spg', 'ts_pct', 'plus_minus']
po_features = (out.rename(columns={'player_id': 'nba_player_id', 'name': 'player_name', **{c: f'po_{c}' for c in po_stats}})
                  [['season', 'nba_player_id', 'player_name', 'name_key', 'last_team', 'age', 'years_in_nba']
                   + [f'po_{c}' for c in po_stats]]
                  .sort_values(['season', 'nba_player_id']).reset_index(drop=True))
print('ts_pct set to blank:', po_features['po_ts_pct'].isna().sum())
po_features.head()

ts_pct set to blank: 66


,season,nba_player_id,player_name,name_key,last_team,age,years_in_nba,po_gp,po_mpg,po_ppg,po_rpg,po_apg,po_bpg,po_spg,po_ts_pct,po_plus_minus
0,2016,101106,Andrew Bogut,andrew bogut,GSW,31.0,11,22,16.606818,4.590909,5.681818,1.363636,1.590909,0.636364,0.607,1.545455
1,2016,101107,Marvin Williams,marvin williams,CHA,30.0,11,7,32.512143,5.142857,6.857143,0.857143,0.428571,0.857143,0.341,-9.571429
2,2016,101108,Chris Paul,chris paul,LAC,31.0,11,4,31.323333,23.750000,4.000000,7.250000,0.000000,2.250000,0.575,7.500000
3,2016,101109,Raymond Felton,raymond felton,DAL,32.0,11,5,34.464667,15.000000,4.600000,4.600000,0.000000,1.200000,0.508,-11.400000
4,2016,101111,Charlie Villanueva,charlie villanueva,DAL,31.0,11,4,5.093750,2.250000,0.500000,0.250000,0.000000,0.000000,0.349,-2.750000


## 4. Validate the final table

In [13]:
f = po_features
checks = {
    'one row per season + player': not f.duplicated(['season', 'nba_player_id']).any(),
    f'seasons are {FIRST_SEASON}-{LAST_SEASON}': f['season'].between(FIRST_SEASON, LAST_SEASON).all() and f['season'].nunique() == 11,
    'every row has a name, age, and years in the NBA': f[['player_name', 'name_key', 'age', 'years_in_nba']].notna().all().all(),
    'games played is 1-28': f['po_gp'].between(1, 28).all(),
    'ts_pct is blank only when there were no attempts': f['po_ts_pct'].isna().sum() == ((out['fga'] + out['fta']) == 0).sum(),
}
for name, ok in checks.items():
    print('PASS' if ok else 'FAIL', '-', name)
assert all(checks.values())
f.drop(columns=['season']).describe().T[['count', 'mean', 'min', '50%', 'max']].round(2)

PASS - one row per season + player
PASS - seasons are 2016-2026
PASS - every row has a name, age, and years in the NBA
PASS - games played is 1-28
PASS - ts_pct is blank only when there were no attempts


,count,mean,min,50%,max
age,2405.0,27.20,19.0,27.00,43.00
years_in_nba,2405.0,6.42,1.0,6.00,23.00
po_gp,2405.0,8.20,1.0,6.00,24.00
po_mpg,2405.0,19.37,0.0,18.31,44.58
po_ppg,2405.0,8.38,0.0,6.15,37.40
po_rpg,2405.0,3.42,0.0,2.73,16.33
po_apg,2405.0,1.76,0.0,1.00,12.22
po_bpg,2405.0,0.37,0.0,0.22,4.00
po_spg,2405.0,0.57,0.0,0.45,3.50
po_ts_pct,2339.0,0.53,0.0,0.55,1.50


**What we found:** Every check passes. The only blanks are `po_ts_pct` for the 66 players with no playoff shot attempts, and `last_team` for the 75 rows where the source has none.

## 5. Save

In [14]:
out_path = PROCESSED / 'llimllib_player_playoff_features.csv'
po_features.to_csv(out_path, index=False)
print('saved', po_features.shape, 'to', out_path)

saved (2405, 16) to ..\..\data-files\02-processed\llimllib_player_playoff_features.csv


**Output:** `data-files/02-processed/llimllib_player_playoff_features.csv`, **one row per player + postseason** (only players who appeared in a playoff game, 2015-16 to 2025-26). Keys: `season` + `nba_player_id`. Notebook `999` joins it onto the regular-season table; players without a playoff row get blank `po_*` columns there.